# Step 4 — Analysis & Visualization

Produces, from the results saved by Step 3:

1. **Combined ROC plots** — one figure per evaluation protocol, with **only the classifiers you choose** overlaid (configured by `ROC_MODELS`)
2. **Decision-boundary visualizations** — UMAP projection with each model's confidence (one figure per classifier in `ANALYSIS_MODELS`)
3. **SHAP analysis** with **per-classifier dispatch**:
   - **XGBoost** → `shap.TreeExplainer` (fast, exact)
   - **TabNet** → built-in attention-based importance
   - **All PyTorch models** → `shap.GradientExplainer`
4. Tidy comparison tables

All outputs land in `WORK_DIR/plots/`.

## Cell 1 — Configuration

**`ROC_MODELS`** controls which classifiers appear in the combined ROC plots. Set to `None` to use all available. Set to a list of names to filter.

In [ ]:
WORK_DIR = '/content/drive/MyDrive/pscv_work'

# ============================================================
# ROC SELECTION
# ============================================================
# Which classifiers' ROC curves to overlay in the combined ROC plots.
# Set to None for "use everything in results.pkl".
# Otherwise, give a list of classifier names.
#
# Examples:
#   ROC_MODELS = None                                    # all available
#   ROC_MODELS = ['MLP', 'CNN', 'XGBoost']                # just these 3
#   ROC_MODELS = ['XGBoost', 'TabNet', 'FT-Transformer']  # only tabular
#   ROC_MODELS = ['BlockMoE', 'WideDeep', 'BayesianMLP']  # only block-aware
ROC_MODELS = None
# ============================================================

# Which classifiers to generate decision-boundary + SHAP plots for.
# Boundary + SHAP are expensive — restrict to the most interesting models.
# Set to None for "all available" (slow), or a list to filter.
ANALYSIS_MODELS = None

# Plot styling
ROC_LINE_WIDTH = 2.0
ROC_LEGEND_FONTSIZE = 9

import os, json, pickle
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from typing import Optional, List
from dataclasses import dataclass
from abc import ABC, abstractmethod

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

PLOTS_DIR = os.path.join(WORK_DIR, 'plots')
ROC_DIR   = os.path.join(PLOTS_DIR, 'roc')
BND_DIR   = os.path.join(PLOTS_DIR, 'boundary')
SHAP_DIR  = os.path.join(PLOTS_DIR, 'shap')
for d in [PLOTS_DIR, ROC_DIR, BND_DIR, SHAP_DIR]:
    os.makedirs(d, exist_ok=True)

with open(os.path.join(WORK_DIR, 'results.pkl'), 'rb') as f:
    all_results = pickle.load(f)

d = np.load(os.path.join(WORK_DIR, 'features.npz'), allow_pickle=True)
X_train = d['X_train']; y_train = d['y_train'].astype(np.float32)
X_indep = d['X_indep']; y_indep = d['y_indep'].astype(np.float32)
block_sizes = d['block_sizes'].tolist()
block_names = [str(n) for n in d['block_names']]

available_models = list(all_results)
print('Models with results :', available_models)

if ROC_MODELS is None:
    ROC_MODELS = available_models
if ANALYSIS_MODELS is None:
    ANALYSIS_MODELS = available_models

# Filter to ones actually present
ROC_MODELS = [m for m in ROC_MODELS if m in available_models]
ANALYSIS_MODELS = [m for m in ANALYSIS_MODELS if m in available_models]

print()
print('ROC plots will feature   :', ROC_MODELS)
print('Boundary + SHAP plots for:', ANALYSIS_MODELS)

## Cell 2 — Install plotting dependencies

In [ ]:
!pip install -q shap umap-learn matplotlib seaborn

# Also need to re-import classifier dependencies for Step 4 SHAP work
HAVE_XGB = HAVE_TABNET = HAVE_FTT = False
try:
    import xgboost; HAVE_XGB = True
except ImportError:
    print('xgboost not available — XGBoost SHAP will be skipped')
try:
    import pytorch_tabnet; HAVE_TABNET = True
except ImportError:
    print('pytorch-tabnet not available — TabNet SHAP/reload will be skipped')
try:
    import rtdl_revisiting_models; HAVE_FTT = True
except ImportError:
    print('rtdl_revisiting_models not available — FT-Transformer SHAP/reload will be skipped')

## Cell 3 — Combined ROC plot, **filtered by `ROC_MODELS`**

For each evaluation protocol, exactly the classifiers in `ROC_MODELS` are overlaid.
For CV protocols, predictions are pooled across folds before computing the ROC.

In [ ]:
from sklearn.metrics import roc_curve, auc as sk_auc


def gather_roc_data(model_results, protocol):
    if protocol not in model_results:
        return None
    data = model_results[protocol]
    if protocol in ('cv5', 'cv10'):
        y_true = np.concatenate([f['y_true'] for f in data['folds']])
        y_prob = np.concatenate([f['y_prob'] for f in data['folds']])
    else:
        y_true = data['y_true']
        y_prob = data['y_prob']
    return y_true, y_prob


def plot_combined_roc(protocol, out_path, model_list):
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    plotted = 0
    for mname in model_list:
        if mname not in all_results:
            continue
        rd = gather_roc_data(all_results[mname], protocol)
        if rd is None:
            continue
        y_true, y_prob = rd
        if len(np.unique(y_true)) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        a = sk_auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=ROC_LINE_WIDTH, label=f'{mname} (AUC = {a:.3f})')
        plotted += 1
    if plotted == 0:
        plt.close(fig)
        print(f'  no data for {protocol}, skipped')
        return
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Combined ROC — {protocol}  ({plotted} classifiers)')
    ax.set_xlim(-0.01, 1.0); ax.set_ylim(0.0, 1.01)
    ax.legend(loc='lower right', fontsize=ROC_LEGEND_FONTSIZE)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f'  wrote {out_path}')


for proto in ['self_consistency', 'independent', 'cv5', 'cv10']:
    out = os.path.join(ROC_DIR, f'roc_{proto}.png')
    plot_combined_roc(proto, out, ROC_MODELS)

## Cell 4 — Display ROC figures inline

In [ ]:
from IPython.display import Image, display

for proto in ['self_consistency', 'independent', 'cv5', 'cv10']:
    p = os.path.join(ROC_DIR, f'roc_{proto}.png')
    if os.path.exists(p):
        print(proto)
        display(Image(p))

## Cell 5 — Per-classifier ROC table

Quick textual summary of AUC for each classifier × protocol that's actually featured.

In [ ]:
roc_rows = []
for mname in ROC_MODELS:
    row = {'model': mname}
    for proto in ['self_consistency', 'independent', 'cv5', 'cv10']:
        rd = gather_roc_data(all_results.get(mname, {}), proto)
        if rd is None:
            row[proto] = '-'
        else:
            y_true, y_prob = rd
            if len(np.unique(y_true)) < 2:
                row[proto] = '-'
            else:
                fpr, tpr, _ = roc_curve(y_true, y_prob)
                row[proto] = f'{sk_auc(fpr, tpr):.4f}'
    roc_rows.append(row)

print('AUC for ROC-featured classifiers:')
print(pd.DataFrame(roc_rows).to_string(index=False))

## Cell 6 — Re-define classifier classes so pickled checkpoints can be loaded

For decision-boundary and SHAP we need to **load** the trained classifiers saved by Step 3. Python pickle requires the class definitions in scope at unpickling time.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset


# ---------- FeatureCombiner (same as Step 3) ----------
class FeatureCombiner:
    def __init__(self, block_sizes, block_names):
        self.block_sizes = list(block_sizes)
        self.block_names = list(block_names)
        self.scalers = [StandardScaler() for _ in self.block_sizes]
        self.fitted = False
    @property
    def block_slices(self):
        s, start = [], 0
        for b in self.block_sizes:
            s.append(slice(start, start + b)); start += b
        return s
    def fit(self, X):
        for sl, sc in zip(self.block_slices, self.scalers): sc.fit(X[:, sl])
        self.fitted = True; return self
    def transform(self, X):
        out = np.empty_like(X, dtype=np.float32)
        for sl, sc in zip(self.block_slices, self.scalers):
            out[:, sl] = sc.transform(X[:, sl])
        return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
    def fit_transform(self, X): return self.fit(X).transform(X)


# ---------- TrainConfig ----------
@dataclass
class TrainConfig:
    epochs: int = 60
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-5
    early_stopping_patience: int = 10
    device: str = DEVICE
    pos_weight: float = 1.0


# ---------- Deep nn.Modules (same as Step 3 Cell 6) ----------
class MLPClassifier(nn.Module):
    def __init__(self, in_dim, hidden=(512, 256, 128), dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)


class CNN1DClassifier(nn.Module):
    def __init__(self, in_dim, channels=(64, 128, 64), kernel=5, dropout=0.3):
        super().__init__()
        layers, c_prev = [], 1
        for c in channels:
            layers += [nn.Conv1d(c_prev, c, kernel_size=kernel, padding=kernel//2),
                       nn.ReLU(), nn.BatchNorm1d(c), nn.MaxPool1d(2), nn.Dropout(dropout)]
            c_prev = c
        self.conv = nn.Sequential(*layers)
        out_len = in_dim
        for _ in channels: out_len //= 2
        self.head = nn.Sequential(nn.Linear(c_prev * out_len, 128), nn.ReLU(),
                                   nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        z = self.conv(x.unsqueeze(1)).flatten(1)
        return self.head(z).squeeze(-1)


class _SequentialLSTM(nn.Module):
    def __init__(self, in_dim, hidden_size=128, num_layers=2,
                 bidirectional=False, dropout=0.3):
        super().__init__()
        self.bidir = bidirectional
        self.embed = nn.Linear(1, 32)
        self.lstm = nn.LSTM(32, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0,
                            bidirectional=bidirectional)
        feat = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(nn.Linear(feat, 64), nn.ReLU(),
                                   nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        e = self.embed(x.unsqueeze(-1))
        _, (h, _) = self.lstm(e)
        last = torch.cat([h[-2], h[-1]], dim=-1) if self.bidir else h[-1]
        return self.head(last).squeeze(-1)


class LSTMClassifier(_SequentialLSTM):
    def __init__(self, in_dim, **kw): super().__init__(in_dim, bidirectional=False, **kw)


class BiLSTMClassifier(_SequentialLSTM):
    def __init__(self, in_dim, **kw): super().__init__(in_dim, bidirectional=True, **kw)


class CNNBiLSTMClassifier(nn.Module):
    def __init__(self, in_dim, conv_channels=64, hidden_size=128, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, conv_channels, 7, padding=3), nn.ReLU(),
            nn.BatchNorm1d(conv_channels), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(conv_channels, conv_channels, 5, padding=2), nn.ReLU(),
            nn.BatchNorm1d(conv_channels), nn.MaxPool1d(2), nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(conv_channels, hidden_size, 2,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(2*hidden_size, 128), nn.ReLU(),
                                   nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        z = self.conv(x.unsqueeze(1)).transpose(1, 2)
        _, (h, _) = self.lstm(z)
        last = torch.cat([h[-2], h[-1]], dim=-1)
        return self.head(last).squeeze(-1)


class TransformerClassifier(nn.Module):
    def __init__(self, in_dim, token_size=16, d_model=128,
                 nhead=4, num_layers=3, dropout=0.2):
        super().__init__()
        self.token_size = token_size
        self.pad = (token_size - in_dim % token_size) % token_size
        self.n_tokens = (in_dim + self.pad) // token_size
        self.proj = nn.Linear(token_size, d_model)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.zeros(1, self.n_tokens + 1, d_model))
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, 4*d_model,
                                                dropout, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)
    def forward(self, x):
        if self.pad: x = nn.functional.pad(x, (0, self.pad))
        b = x.size(0); x = x.view(b, self.n_tokens, self.token_size)
        z = self.proj(x)
        cls = self.cls.expand(b, -1, -1)
        z = torch.cat([cls, z], dim=1) + self.pos
        z = self.encoder(z)
        return self.head(self.norm(z[:, 0])).squeeze(-1)


MODEL_REGISTRY = {
    'MLP': MLPClassifier, 'CNN': CNN1DClassifier,
    'LSTM': LSTMClassifier, 'BiLSTM': BiLSTMClassifier,
    'CNN-BiLSTM': CNNBiLSTMClassifier, 'Transformer': TransformerClassifier,
}


def build_nn_model(name, in_dim, **kwargs):
    return MODEL_REGISTRY[name](in_dim=in_dim, **kwargs)


# ---------- Classifier ABC + NNClassifier wrapper ----------
class Classifier(ABC):
    name: str = 'base'
    @abstractmethod
    def fit(self, X_tr, y_tr, X_val, y_val): ...
    @abstractmethod
    def predict_proba(self, X): ...
    def torch_module(self): return None
    def feature_importance(self): return None


class NNClassifier(Classifier):
    def __init__(self, model_name, in_dim, train_cfg, model_kwargs=None):
        self.name = model_name
        self.in_dim = in_dim
        self.cfg = train_cfg
        self.model_kwargs = model_kwargs or {}
        self.model = None
    def _factory(self):
        return build_nn_model(self.name, self.in_dim, **self.model_kwargs)
    def fit(self, X_tr, y_tr, X_val, y_val):
        # Simplified for reload-then-fit if needed; full version is in Step 3
        model = self._factory().to(self.cfg.device)
        opt = torch.optim.AdamW(model.parameters(), lr=self.cfg.lr,
                                weight_decay=self.cfg.weight_decay)
        loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([self.cfg.pos_weight], device=self.cfg.device))
        ds = TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).float())
        loader = DataLoader(ds, batch_size=self.cfg.batch_size, shuffle=True)
        best_auc, best_state, stale = -1.0, None, 0
        X_val_t = torch.from_numpy(X_val).float().to(self.cfg.device)
        for ep in range(self.cfg.epochs):
            model.train()
            for xb, yb in loader:
                xb, yb = xb.to(self.cfg.device), yb.to(self.cfg.device)
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                opt.step()
            model.eval()
            with torch.no_grad():
                pv = torch.sigmoid(model(X_val_t)).cpu().numpy()
            try: auc = roc_auc_score(y_val, pv)
            except Exception: auc = 0.0
            if auc > best_auc:
                best_auc = auc; stale = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                stale += 1
                if stale >= self.cfg.early_stopping_patience: break
        if best_state is not None: model.load_state_dict(best_state)
        self.model = model
        return self
    @torch.no_grad()
    def predict_proba(self, X):
        self.model.eval()
        out = []
        for i in range(0, len(X), 256):
            xb = torch.from_numpy(X[i:i+256]).float().to(self.cfg.device)
            out.append(torch.sigmoid(self.model(xb)).cpu().numpy())
        return np.concatenate(out)
    def torch_module(self): return self.model


# ---------- Exotic classifier classes (XGBoost, TabNet, FTT, TabResNet, BlockMoE, WideDeep, BayesianMLP) ----------
class XGBoostClassifier(Classifier):
    name = 'XGBoost'
    def __init__(self, in_dim, train_cfg):
        self.in_dim = in_dim; self.cfg = train_cfg; self.model = None
    def fit(self, X_tr, y_tr, X_val, y_val): raise NotImplementedError('Reload only')
    def predict_proba(self, X): return self.model.predict_proba(X)[:, 1]
    def feature_importance(self):
        return None if self.model is None else self.model.feature_importances_


class TabNetClassifier_(Classifier):
    name = 'TabNet'
    def __init__(self, in_dim, train_cfg):
        self.in_dim = in_dim; self.cfg = train_cfg; self.model = None
    def fit(self, X_tr, y_tr, X_val, y_val): raise NotImplementedError('Reload only')
    def predict_proba(self, X): return self.model.predict_proba(X)[:, 1]
    def feature_importance(self):
        return None if self.model is None else self.model.feature_importances_


class _FTWrap(nn.Module):
    def __init__(self, ft): super().__init__(); self.ft = ft
    def forward(self, x): return self.ft(x, None).squeeze(-1)


class FTTransformerClassifier(NNClassifier):
    name = 'FT-Transformer'
    def __init__(self, in_dim, train_cfg):
        self.name = 'FT-Transformer'; self.in_dim = in_dim
        self.cfg = train_cfg; self.model_kwargs = {}; self.model = None
    def _factory(self):
        from rtdl_revisiting_models import FTTransformer
        kwargs = FTTransformer.get_default_kwargs()
        return _FTWrap(FTTransformer(
            n_cont_features=self.in_dim, cat_cardinalities=[], d_out=1, **kwargs))


class _ResNetBlock(nn.Module):
    def __init__(self, d, d_hidden, dropout):
        super().__init__()
        self.norm = nn.BatchNorm1d(d)
        self.lin1 = nn.Linear(d, d_hidden); self.act = nn.GELU()
        self.drop1 = nn.Dropout(dropout)
        self.lin2 = nn.Linear(d_hidden, d); self.drop2 = nn.Dropout(dropout)
    def forward(self, x):
        z = self.norm(x); z = self.act(self.lin1(z)); z = self.drop1(z)
        z = self.lin2(z); z = self.drop2(z)
        return x + z


class TabResNet(nn.Module):
    def __init__(self, in_dim, d=256, d_hidden=512, n_blocks=4, dropout=0.2):
        super().__init__()
        self.proj = nn.Linear(in_dim, d)
        self.blocks = nn.Sequential(*[_ResNetBlock(d, d_hidden, dropout) for _ in range(n_blocks)])
        self.norm = nn.BatchNorm1d(d); self.head = nn.Linear(d, 1)
    def forward(self, x):
        z = self.proj(x); z = self.blocks(z); z = self.norm(z)
        return self.head(z).squeeze(-1)


class TabResNetClassifier(NNClassifier):
    name = 'TabResNet'
    def __init__(self, in_dim, train_cfg):
        self.name = 'TabResNet'; self.in_dim = in_dim
        self.cfg = train_cfg; self.model_kwargs = {}; self.model = None
    def _factory(self): return TabResNet(self.in_dim)


class _Expert(nn.Module):
    def __init__(self, d_in, d_hidden=256, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.GELU(),
            nn.BatchNorm1d(d_hidden), nn.Dropout(dropout),
            nn.Linear(d_hidden, d_hidden // 2), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden // 2, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)


class BlockMoE(nn.Module):
    def __init__(self, block_sizes, d_hidden=256, dropout=0.2):
        super().__init__()
        self.block_sizes = block_sizes; self.slices = []
        s = 0
        for bs in block_sizes:
            self.slices.append(slice(s, s+bs)); s += bs
        self.experts = nn.ModuleList([
            _Expert(bs, d_hidden=d_hidden, dropout=dropout) for bs in block_sizes
        ])
        gate_in = 2 * len(block_sizes)
        self.gate = nn.Sequential(
            nn.Linear(gate_in, 32), nn.GELU(),
            nn.Linear(32, len(block_sizes)),
        )
    def forward(self, x):
        stats = []
        for sl in self.slices:
            xb = x[:, sl]
            stats.append(xb.mean(dim=1, keepdim=True))
            stats.append(xb.std(dim=1, keepdim=True))
        s = torch.cat(stats, dim=1)
        weights = torch.softmax(self.gate(s), dim=1)
        expert_logits = torch.stack(
            [exp(x[:, sl]) for exp, sl in zip(self.experts, self.slices)], dim=1)
        return (weights * expert_logits).sum(dim=1)


class BlockMoEClassifier(NNClassifier):
    name = 'BlockMoE'
    def __init__(self, in_dim, train_cfg, block_sizes):
        self.name = 'BlockMoE'; self.in_dim = in_dim
        self.cfg = train_cfg; self.block_sizes = block_sizes
        self.model_kwargs = {}; self.model = None
    def _factory(self): return BlockMoE(self.block_sizes)


class WideDeep(nn.Module):
    def __init__(self, block_sizes, deep_hidden=(512, 256, 128), dropout=0.3):
        super().__init__()
        self.wide_dim = block_sizes[0]
        self.deep_dim = sum(block_sizes[1:])
        self.wide_slice = slice(0, self.wide_dim)
        self.deep_slice = slice(self.wide_dim, self.wide_dim + self.deep_dim)
        self.wide = nn.Linear(self.wide_dim, 1)
        layers, prev = [], self.deep_dim
        for h in deep_hidden:
            layers += [nn.Linear(prev, h), nn.GELU(),
                       nn.BatchNorm1d(h), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.deep = nn.Sequential(*layers)
    def forward(self, x):
        w = self.wide(x[:, self.wide_slice]).squeeze(-1)
        d = self.deep(x[:, self.deep_slice]).squeeze(-1)
        return w + d


class WideDeepClassifier(NNClassifier):
    name = 'WideDeep'
    def __init__(self, in_dim, train_cfg, block_sizes):
        self.name = 'WideDeep'; self.in_dim = in_dim
        self.cfg = train_cfg; self.block_sizes = block_sizes
        self.model_kwargs = {}; self.model = None
    def _factory(self): return WideDeep(self.block_sizes)


class BayesianMLP(nn.Module):
    def __init__(self, in_dim, hidden=(512, 256, 128), dropout=0.25):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)


class BayesianMLPClassifier(NNClassifier):
    name = 'BayesianMLP'
    def __init__(self, in_dim, train_cfg, T=30):
        self.name = 'BayesianMLP'; self.in_dim = in_dim
        self.cfg = train_cfg; self.T = T
        self.model_kwargs = {}; self.model = None; self._last_std = None
    def _factory(self): return BayesianMLP(self.in_dim)
    @torch.no_grad()
    def predict_proba(self, X):
        for m in self.model.modules():
            if isinstance(m, nn.Dropout): m.train()
            else: m.eval()
        probs_T = []
        for _ in range(self.T):
            out = []
            for i in range(0, len(X), 256):
                xb = torch.from_numpy(X[i:i+256]).float().to(self.cfg.device)
                out.append(torch.sigmoid(self.model(xb)).cpu().numpy())
            probs_T.append(np.concatenate(out))
        probs_T = np.stack(probs_T, axis=0)
        mean = probs_T.mean(axis=0)
        self._last_std = probs_T.std(axis=0)
        return mean
    def predictive_std(self): return self._last_std


print('All 13 classifier classes redefined (for unpickling Step 3 checkpoints).')

## Cell 7 — Load each saved (classifier, combiner) pair from Step 3

In [ ]:
def load_clf(mname):
    p = os.path.join(WORK_DIR, f'clf_{mname}_independent.pkl')
    if not os.path.exists(p):
        return None, None
    try:
        with open(p, 'rb') as f:
            d = pickle.load(f)
        return d['classifier'], d['combiner']
    except Exception as e:
        print(f'  {mname}: load failed ({type(e).__name__}: {e})')
        return None, None


print('Reload status:')
for m in ANALYSIS_MODELS:
    clf, comb = load_clf(m)
    status = 'OK' if clf is not None else 'MISSING'
    print(f'  {m:18s}: {status}')

## Cell 8 — Decision-boundary visualization (UMAP)

For each classifier in `ANALYSIS_MODELS` that we could reload, project train + indep-test to 2-D, KNN-regress the model's probability onto a grid, and plot side-by-side.

In [ ]:
import umap
from sklearn.neighbors import KNeighborsRegressor
from matplotlib.colors import ListedColormap


def plot_decision_boundary(mname, out_path, grid_size=200, k_neighbors=15):
    clf, combiner = load_clf(mname)
    if clf is None:
        print(f'  {mname}: no reload, skipping boundary')
        return
    try:
        Xtr_s = combiner.transform(X_train)
        Xte_s = combiner.transform(X_indep)
        p_tr = clf.predict_proba(Xtr_s)
        p_te = clf.predict_proba(Xte_s)
    except Exception as e:
        print(f'  {mname}: predict failed: {e}')
        return

    n_tr = len(Xtr_s)
    reducer = umap.UMAP(n_components=2, random_state=0, n_neighbors=15, min_dist=0.1)
    proj_all = reducer.fit_transform(np.vstack([Xtr_s, Xte_s]))
    proj_tr, proj_te = proj_all[:n_tr], proj_all[n_tr:]

    x_min, x_max = proj_all[:, 0].min() - 1, proj_all[:, 0].max() + 1
    y_min, y_max = proj_all[:, 1].min() - 1, proj_all[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_size),
                          np.linspace(y_min, y_max, grid_size))
    grid = np.c_[xx.ravel(), yy.ravel()]

    knn_tr = KNeighborsRegressor(n_neighbors=min(k_neighbors, len(proj_tr))).fit(proj_tr, p_tr)
    knn_te = KNeighborsRegressor(n_neighbors=min(k_neighbors, len(proj_te))).fit(proj_te, p_te)
    zz_tr = knn_tr.predict(grid).reshape(xx.shape)
    zz_te = knn_te.predict(grid).reshape(xx.shape)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
    cmap_pt = ListedColormap(['#3b6fff', '#ff4040'])
    for ax, zz, proj, y, ttl in [
        (axes[0], zz_tr, proj_tr, y_train, 'Training (70%)'),
        (axes[1], zz_te, proj_te, y_indep, 'Independent test (30%)'),
    ]:
        ax.contourf(xx, yy, zz, levels=20, cmap=plt.cm.RdBu_r, alpha=0.6, vmin=0, vmax=1)
        ax.contour(xx, yy, zz, levels=[0.5], colors='black', linewidths=1.2)
        ax.scatter(proj[:, 0], proj[:, 1], c=y, cmap=cmap_pt,
                    s=10, edgecolor='k', linewidth=0.2)
        ax.set_title(f'{ttl}  (n={len(proj)})')
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'{mname} — decision landscape (UMAP)\n'
                  'Background = P(class=1).  Black line = 0.5 boundary.', fontsize=12)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f'  wrote {out_path}')


for mname in ANALYSIS_MODELS:
    print(f'\n{mname}:')
    plot_decision_boundary(mname, os.path.join(BND_DIR, f'boundary_{mname}.png'))

## Cell 9 — Display boundary figures inline

In [ ]:
for mname in ANALYSIS_MODELS:
    p = os.path.join(BND_DIR, f'boundary_{mname}.png')
    if os.path.exists(p):
        print(mname)
        display(Image(p))

## Cell 10 — SHAP analysis with **per-classifier dispatch**

| Classifier type | SHAP explainer |
|---|---|
| `XGBoostClassifier` | `shap.TreeExplainer` (fast, exact) |
| `TabNetClassifier_` | built-in attention-based importance, broadcast to per-sample shape |
| Everything else (PyTorch) | `shap.GradientExplainer` |

In [ ]:
import shap


PSCV_GROUPS = {
    'seqMat_raw':     slice(0,   10),
    'seqMat_central': slice(10,  20),
    'seqMat_hahn':    slice(20,  30),
    'freq_vec':       slice(30,  51),
    'PRIM_raw':       slice(51,  61),
    'PRIM_central':   slice(61,  71),
    'PRIM_hahn':      slice(71,  81),
    'AAPIV':          slice(81,  102),
    'RPRIM_raw':      slice(102, 112),
    'RPRIM_central':  slice(112, 122),
    'RPRIM_hahn':     slice(122, 132),
    'RAAPIV':         slice(132, 153),
}


def compute_shap_for_classifier(clf, X_bg, X_ex, n_background=100, n_explain=200, seed=0):
    """Dispatches to the right SHAP explainer based on classifier type."""
    rng = np.random.RandomState(seed)
    bg_idx = rng.choice(len(X_bg), size=min(n_background, len(X_bg)), replace=False)
    ex_idx = rng.choice(len(X_ex), size=min(n_explain,   len(X_ex)),   replace=False)
    bg = X_bg[bg_idx]
    ex = X_ex[ex_idx]

    # --- XGBoost: TreeExplainer ---
    if isinstance(clf, XGBoostClassifier):
        explainer = shap.TreeExplainer(clf.model)
        sv = explainer.shap_values(ex)
        if isinstance(sv, list):
            sv = sv[-1]
        return np.asarray(sv)

    # --- TabNet: built-in feature importance (broadcast to per-sample shape) ---
    if isinstance(clf, TabNetClassifier_):
        fi = clf.feature_importance()
        if fi is None:
            return np.zeros_like(ex)
        fi = fi / (fi.sum() + 1e-12)
        return np.tile(fi[None, :], (ex.shape[0], 1)) * ex

    # --- PyTorch (NNClassifier and subclasses): GradientExplainer ---
    module = clf.torch_module()
    if module is None:
        raise TypeError(f'No SHAP path for {type(clf).__name__}')

    class ProbWrap(nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, x): return torch.sigmoid(self.m(x))

    wrapped = ProbWrap(module).to(DEVICE).eval()
    bg_t = torch.from_numpy(bg).float().to(DEVICE)
    ex_t = torch.from_numpy(ex).float().to(DEVICE)
    explainer = shap.GradientExplainer(wrapped, bg_t)
    vals = explainer.shap_values(ex_t)
    if isinstance(vals, list): vals = vals[0]
    if vals.ndim == 3 and vals.shape[-1] == 1: vals = vals[..., 0]
    return np.asarray(vals)


def plot_block_importance(shap_vals, mname, out_path):
    abs_s = np.abs(shap_vals)
    importance, start = {}, 0
    for name, size in zip(block_names, block_sizes):
        importance[name] = float(abs_s[:, start:start+size].mean())
        start += size
    names = list(importance)
    vals = [importance[n] for n in names]
    order = np.argsort(vals)
    fig, ax = plt.subplots(figsize=(7, max(2.5, 0.5 * len(names))))
    ax.barh([names[i] for i in order], [vals[i] for i in order], color='#1f77b4')
    ax.set_xlabel('mean(|SHAP|)')
    ax.set_title(f'Feature-block importance — {mname}')
    fig.tight_layout(); fig.savefig(out_path, dpi=200); plt.close(fig)
    print(f'  wrote {out_path}')


def plot_pscv_topk(shap_vals, mname, out_path, top_k=30):
    pscv_slice = slice(0, block_sizes[0])
    pscv_shap = shap_vals[:, pscv_slice]
    mean_abs = np.abs(pscv_shap).mean(axis=0)
    labels = [''] * pscv_shap.shape[1]
    for gname, gslice in PSCV_GROUPS.items():
        for k, idx in enumerate(range(gslice.start, gslice.stop)):
            labels[idx] = f'{gname}[{k}]'
    top_idx = np.argsort(mean_abs)[::-1][:top_k]
    fig, ax = plt.subplots(figsize=(9, max(4, 0.3 * top_k)))
    ax.barh([labels[i] for i in top_idx][::-1],
             mean_abs[top_idx][::-1], color='#ff7f0e')
    ax.set_xlabel('mean(|SHAP|)')
    ax.set_title(f'Top-{top_k} PSCV features — {mname}')
    fig.tight_layout(); fig.savefig(out_path, dpi=200); plt.close(fig)
    print(f'  wrote {out_path}')


for mname in ANALYSIS_MODELS:
    print(f'\n--- SHAP for {mname} ---')
    clf, combiner = load_clf(mname)
    if clf is None:
        print('  no reload, skipping')
        continue
    try:
        Xtr_s = combiner.transform(X_train)
        Xte_s = combiner.transform(X_indep)
        shap_vals = compute_shap_for_classifier(
            clf, Xtr_s, Xte_s, n_background=100, n_explain=min(200, len(Xte_s)))
        plot_block_importance(shap_vals, mname,
                              os.path.join(SHAP_DIR, f'shap_blocks_{mname}.png'))
        plot_pscv_topk(shap_vals, mname,
                        os.path.join(SHAP_DIR, f'shap_pscv_{mname}.png'))
    except Exception as e:
        print(f'  SHAP failed: {type(e).__name__}: {e}')

## Cell 11 — Display SHAP figures inline

In [ ]:
for mname in ANALYSIS_MODELS:
    for kind in ['blocks', 'pscv']:
        p = os.path.join(SHAP_DIR, f'shap_{kind}_{mname}.png')
        if os.path.exists(p):
            print(f'{mname} — {kind}')
            display(Image(p))

## Cell 12 — Final summary table

In [ ]:
summary_df = pd.read_csv(os.path.join(WORK_DIR, 'summary.csv'))
print('Full results table:')
print(summary_df.to_string(index=False))

print('\nAUC pivot:')
auc_table = summary_df.pivot(index='model', columns='protocol', values='AUC')
proto_order = [c for c in ['self_consistency', 'independent', 'cv5', 'cv10'] if c in auc_table.columns]
auc_table = auc_table[proto_order]
print(auc_table.to_string())

print(f'\nAll plots saved under: {PLOTS_DIR}')
print('Pipeline complete.')